In [1]:
import sys
print(sys.executable)

/opt/gpudata/shaowenz/projects/JointDiff/env/jointdiff/bin/python3


In [2]:
import os
import shutil
import argparse
from tqdm.auto import tqdm
from easydict import EasyDict

import torch
import torch.utils.tensorboard
from torch.nn.utils import clip_grad_norm_
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.multiprocessing

import importlib
import sys
sys.path.append("../")

import jointdiff
importlib.reload(jointdiff)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.multiprocessing.set_sharing_strategy('file_system')
torch.autograd.set_detect_anomaly(True)

# Configuration

In [3]:
def dict_print(d, indent = ''):
    if isinstance(d, dict) or isinstance(d, EasyDict):
        for key in d:
            if isinstance(d[key], dict) or isinstance(d[key], EasyDict):
                print(indent, key)
                dict_print(d[key], indent = ' ' * 4 + indent)
            else:
                print(indent, key, ' ', d[key])
    else:
        print(indent, d)

In [4]:
config_path = "../../configs/jointdiff-x_dim-128-64-4_step100_lr1.e-4_wd0._posiscale50.0.yml"
args_dict, args_name = load_config(config_path)

In [5]:
dict_print(args_dict)

 model
     type   diffab
     res_feat_dim   128
     pair_feat_dim   64
     diffusion
         num_steps   100
         eps_net_opt
             num_layers   6
             decoder_version   mlp
         position_scale   [50.0]
         seq_diff_version   multinomial
         seq_model_opt
             d_embedding   8
             d_model   1024
             activation   gelu
             slim   True
             n_layers   16
             kernel_size   5
             r   128
             causal   False
             dropout   0.0
             tie_weights   False
             final_ln   False
         remember_padding   False
         reweighting_term   0.001
         modality   joint
         ps_adapt_scale   1.0
     train_structure   True
     train_sequence   True
     gen_version   complete_gen
     encode_share   True
     train_version   gt
 train
     loss_weights
         rot   1.0
         pos   1.0
         seq   1.0
     max_iters   1000000
     val_freq   1000
     batch

# Data

### Required Features
* **aa:** encoded amino acid sequence, (N, L)
* **pos_heavyatom:** coordinates of heavy atoms, (N, L, 15,3)
* **mask:** True for valid position rather than padding, (N, L)
* **mask_heavyatom:** True for existing atoms, (N, L 15)
* **chain_nb:** chain index, integers, (N, L)
* **res_nb:** residue index (telling which parts are inconsecutive), integers, (N, L) \
------------------------- optional ------------------------------
* **mask_gen:** optional, True for target region to be generated, (N, L)
* **mask_gen_seq:** optional, True for target sequence region to be generated, (N, L)
* **fragment_type:** optional, indicating the type of the residue/position, integers, (N, L)

In [47]:
def tensor_print(t, shape = None, showlen = 6):
    if len(t.shape) == 1:
        if shape is None:
            shape = str(t.shape)
        return f"{shape:<30} %s" % t[:showlen]
    elif shape is None:
        return tensor_print(t[0], shape = str(t.shape), showlen = showlen)
    else:
        return tensor_print(t[0], shape = shape, showlen = showlen)

def batch_print(batch):
    key_list = sorted(batch)
    
    for key in key_list:
        if torch.is_tensor(batch[key]):
            continue
        val_type = str(type(batch[key]))
        if key == 'chain_id' or key == 'icode':
            print(f"{key:<15} {val_type:<23}", len(batch[key]))
        else:
            print(f"{key:<15} {val_type:<23}", batch[key])
    print('*' * 60)
    
    for key in key_list:
        if torch.is_tensor(batch[key]):
            val_type = str(type(batch[key]))
            print(f"{key:<15} {val_type:<23} ", tensor_print(batch[key]))

## Monomer (CATH) Dataset

In [4]:
importlib.reload(jointdiff)
from jointdiff.dataset import SingleChainDataset
from jointdiff.modules.data.data_utils import PaddingCollate

In [40]:
################### configuration ###################
data_dir = "../../data/"
data_cfg = dict(
    ### path to the basic data information
    summary_path = os.path.join(data_dir, "cath_summary_all.tsv"),
    ### path to the raw PDB files (not necessary if the data has been processed)
    pdb_dir = data_dir,
    ### path to the processed data
    processed_dir = data_dir,
    split = 'train',            # dateset
    transform = None,           # transformation 
    reset = False,              # whether reprocess the data (will cover the processed data)
)

################### data loader ###################
dataset = SingleChainDataset(**data_cfg)
dataloader = DataLoader(
    dataset,
    batch_size = 4,
    collate_fn = PaddingCollate(),
    shuffle = True,
    num_workers = 4,
)
for idx, batch_cath in tqdm(enumerate(dataloader)):
    print(idx, batch_cath['id'])
    if idx == 5:
        break

Loading entries:   0%|                           | 0/52110 [00:00<?, ?it/s]

45995 samples loaded for the train set.


0it [00:00, ?it/s]

0 ['2XDA_A1-150', '1HA6_A1-70', '3PMO_A-16-100', '1EN7_A1-80']
1 ['2X12_A236-425', '1BKL_A3-112', '2IVM_A6-57', '3KH8_A165-297']
2 ['2BKY_A9-97', '3OFN_K21-95', '1DFO_A281-417', '4NJ7_F1039-1125']
3 ['5T1G_A108-168', '2LRW_A1-78', '4ZYK_B1-107', '1VEX_A611-666']
4 ['1SOX_A3-88', '1QHH_D571-646', '3BUL_A936-995', '5CZX_D1-107']
5 ['3BYW_D38-194', '2LP5_A1-59', '4DCF_A1-122', '2DJQ_A1-68']


In [41]:
batch_print(batch_cath)

chain           <class 'list'>          ['D', 'A', 'A', 'A']
chain_id        <class 'list'>          160
icode           <class 'list'>          160
id              <class 'list'>          ['3BYW_D38-194', '2LP5_A1-59', '4DCF_A1-122', '2DJQ_A1-68']
region          <class 'list'>          ['38-194', '1-59', '1-122', '1-68']
************************************************************
aa              <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([17, 13, 15, 15, 17, 15])
chain_nb        <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([1, 1, 1, 1, 1, 1])
idx             <class 'torch.Tensor'>   torch.Size([4])                tensor([23492, 17474, 34960, 13438])
mask            <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([True, True, True, True, True, True])
mask_heavyatom  <class 'torch.Tensor'>   torch.Size([4, 160, 15])       tensor([True, True, True, True, True, True])
pos_heavyatom   <class 'torch.Tensor'>   torch.Size([4, 160, 15, 3]

## Multimer (MPNN) Dataset

In [35]:
importlib.reload(jointdiff.dataset)
from jointdiff.dataset import ProteinMPNNDataset
from jointdiff.modules.data.data_utils import PaddingCollate

In [42]:
################### configuration ###################
data_dir = "../../data/ProteinMPNN/"
data_cfg = dict(
    ### path to the basic data information
    summary_path = os.path.join(data_dir, "mpnn_data_info.pkl"),
    ### path to the raw PDB files (not necessary if the data has been processed)
    pdb_dir = os.path.join(data_dir, "pdbs"),
    ### path to the processed data 
    processed_dir = data_dir,
    ### path to the interface of the dictionary
    interface_path = os.path.join(data_dir, "interface_dict_all.pt"),
    dset = 'train',            # dateset
    transform = 'default',     # transformation 
    reset = False,             # whether reprocess the data (will cover the processed data)
    reso_threshold = 3.0,      # resolution threshold
    length_min = 20,           # lowerbound of the threshold
    length_max = 800,          # upperbound of the threshold
    with_monomer = True,       # whether incorparate monomers as well
    load_interface = True,     # whether load the interface information
    with_epitope = True,       # whether incoporate the epitope information
    with_bindingsite = True,   # whether incoporate the binding site information
    with_scaffold = True,      # whether incoporate the scaffold infomation
    random_masking = True,     # whether apply random masking
    mask_threshold = 80,       # size of masking
    dimer_only = False         # whether only collect the dimers
)

################### data loader ###################
dataset = ProteinMPNNDataset(**data_cfg)
dataloader = DataLoader(
    dataset,
    batch_size = 4,
    collate_fn = PaddingCollate(),
    shuffle = True,
    num_workers = 4,
)

for idx, batch_mpnn in tqdm(enumerate(dataloader)):
    print(idx, batch_mpnn['name'])
    if idx == 5:
        break

Interface loaded from ../../data/ProteinMPNN/interface_dict_all.pt.


  0%|          | 0/24754 [00:00<?, ?it/s]

144958 samples loaded for the train set.


0it [00:00, ?it/s]

0 ['3o0f', '6da2', '3dc7', '6imq']
1 ['3c3q', '1x82', '3arg', '6uus']
2 ['7of5', '1w2k', '3d3c', '3g9v']
3 ['5r05', '1hsi', '7n1p', '6hwf']
4 ['1kon', '3np0', '2y6s', '4u1z']
5 ['3zsy', '6m90', '5bwr', '6zyf']


In [43]:
batch_print(batch_mpnn)

chain_id        <class 'list'>          720
name            <class 'list'>          ['3zsy', '6m90', '5bwr', '6zyf']
************************************************************
aa              <class 'torch.Tensor'>   torch.Size([4, 720])           tensor([15, 12,  5,  7, 18, 13])
chain_nb        <class 'torch.Tensor'>   torch.Size([4, 720])           tensor([0, 0, 0, 0, 0, 0])
fragment_type   <class 'torch.Tensor'>   torch.Size([4, 720])           tensor([3, 3, 3, 3, 3, 3])
generate_flag   <class 'torch.Tensor'>   torch.Size([4, 720])           tensor([0, 0, 0, 0, 0, 0])
idx             <class 'torch.Tensor'>   torch.Size([4])                tensor([ 28916,  38281, 108843,  62172])
mask            <class 'torch.Tensor'>   torch.Size([4, 720])           tensor([True, True, True, True, True, True])
mask_heavyatom  <class 'torch.Tensor'>   torch.Size([4, 720, 4])        tensor([1, 1, 1, 1], dtype=torch.int32)
pos_heavyatom   <class 'torch.Tensor'>   torch.Size([4, 720, 4, 3])     tensor

## Boltz

In [ ]:
from boltz1_loader import StructureSequenceDataset, build_dataloader



In [ ]:
ds = StructureSequenceDataset(
    "/opt/gpudata/shaowenz/documents/boltz1_rcsb/rcsb_processed_targets",
    sampler="cluster",        # None | "random" | "cluster" | your own Sampler
    cropper="boltz",          # None | "boltz" | your own Cropper
    max_tokens=256, max_atoms=2304,
    samples_per_epoch=100000,
    pad_to_max_tokens=True, pad_to_max_atoms=True,
)
dl = build_dataloader(ds, batch_size=4, num_workers=4)

In [48]:
p = 'dsfvsrdv'
print(set(p))

{'f', 'r', 'd', 'v', 's'}


## Comparison

In [46]:
print("#" * 45, " CATH ", "#" * 45)
batch_print(batch_cath)
print()
print("#" * 45, " MPNN ", "#" * 45)
batch_print(batch_mpnn)

#############################################  CATH  #############################################
chain           <class 'list'>          ['D', 'A', 'A', 'A']
chain_id        <class 'list'>          160
icode           <class 'list'>          160
id              <class 'list'>          ['3BYW_D38-194', '2LP5_A1-59', '4DCF_A1-122', '2DJQ_A1-68']
region          <class 'list'>          ['38-194', '1-59', '1-122', '1-68']
************************************************************
aa              <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([17, 13, 15, 15, 17, 15])
chain_nb        <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([1, 1, 1, 1, 1, 1])
idx             <class 'torch.Tensor'>   torch.Size([4])                tensor([23492, 17474, 34960, 13438])
mask            <class 'torch.Tensor'>   torch.Size([4, 160])           tensor([True, True, True, True, True, True])
mask_heavyatom  <class 'torch.Tensor'>   torch.Size([4, 160, 15])       tensor([True,

# Model

In [ ]:
from jointdiff.model import DiffusionSingleChainDesign
from jointdiff.modules.utils.train import get_optimizer, get_scheduler
from jointdiff.modules.utils.misc import (
    BlackHole, seed_all,
    get_new_log_dir, get_logger, inf_iterator,
)
from jointdiff.modules.data.data_utils import PaddingCollate
from jointdiff.trainer import (
    train, validate, load_config, get_dataset, count_parameters
)

In [49]:
logits = torch.zeros(2, 4)
torch.nn.functional.log_softmax(logits, dim=-1)

tensor([[-1.3863, -1.3863, -1.3863, -1.3863],
        [-1.3863, -1.3863, -1.3863, -1.3863]])

In [52]:
logits[0] = torch.nan

In [54]:
torch.isnan(logits)

tensor([[ True,  True,  True,  True],
        [False, False, False, False]])

# Training

In [47]:
type_embed = nn.Embedding(5, 6, padding_idx=0)

In [48]:
out = type_embed(batch['fragment_type'])

# Inference

In [49]:
l = [1, 1, 1, 4, 5, 3]
import collections

count_dict = collections.Counter(l)
print(count_dict)

Counter({1: 3, 4: 1, 5: 1, 3: 1})


In [50]:
count_dict.values()

dict_values([3, 1, 1, 1])